In [ ]:
# Cell 1 — Environment check
import torch

print(f"torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: CUDA not available — training will be slow on CPU")

In [ ]:
# Cell 2 — Install dependencies
# ASSUMPTION: torch and torchvision are pre-installed on Kaggle GPU images
!pip install ultralytics --quiet

In [ ]:
# Cell 3 — Path setup
# Marine detector data (replaces the old freshwater set). Build it with:
#   python scripts/get_detector_data.py --source all
# DeepFish + OzFish supply marine fish bounding boxes; NOAA "Labeled Fishes in the Wild"
# frames are added as BACKGROUND / hard-negative images (label files empty) to cut false
# positives. Convert all sources to YOLO format (images/ + labels/) before uploading.
import os

# TODO: replace with your Kaggle marine-detection dataset slug/path once uploaded.
DATASET_ROOT = "/kaggle/input/marine-fish-detection"
OUTPUT_DIR = "/kaggle/working/yolo_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"DATASET_ROOT: {DATASET_ROOT}")
print(f"OUTPUT_DIR:   {OUTPUT_DIR}")

In [ ]:
# Cell 4 — Verify dataset
import os

print(os.listdir(DATASET_ROOT))
print("\nContents of train/images (first 5):")
print(os.listdir(f"{DATASET_ROOT}/train/images")[:5])
print(f"\nTotal train images: {len(os.listdir(f'{DATASET_ROOT}/train/images'))}")
print(f"Total valid images: {len(os.listdir(f'{DATASET_ROOT}/valid/images'))}")

In [ ]:
# Cell 5 — Fix data.yaml paths
# Single-class fish/no-fish detector. NOAA background frames have empty label files and
# need no class entry — they only teach the model what "no fish" looks like.
import yaml

fixed_yaml = {
    "train": f"{DATASET_ROOT}/train/images",
    "val":   f"{DATASET_ROOT}/valid/images",
    "test":  f"{DATASET_ROOT}/test/images",
    "nc":    1,
    "names": ["fish"],
}
with open("/kaggle/working/data.yaml", "w") as f:
    yaml.dump(fixed_yaml, f)
print("Fixed data.yaml written (marine fish, single class + background negatives)")

In [ ]:
# Cell 6 — Train
# ASSUMPTION: device=0 is the Kaggle T4/P100 GPU
# VERIFY: batch=16 fits in GPU VRAM at imgsz=640; reduce to 8 if OOM
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # start from COCO-pretrained nano weights
results = model.train(
    data="/kaggle/working/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project=OUTPUT_DIR,
    name="fish_detector",
    patience=10,
    device=0,
)
print("Training complete.")

In [ ]:
# Cell 7 — Verify and report
import os

best_pt = f"{OUTPUT_DIR}/fish_detector/weights/best.pt"
assert os.path.exists(best_pt), f"best.pt not found at {best_pt}"
size_mb = os.path.getsize(best_pt) / 1024 / 1024
print(f"best.pt found, size {size_mb:.1f} MB")
print("Ready to download")

## Cell 8 — Download instructions

```
# To retrieve the trained weights:
#
# 1. Open the Output tab in the Kaggle right panel
# 2. Navigate to: /kaggle/working/yolo_outputs/fish_detector/weights/
# 3. Download best.pt
# 4. Place locally at: weights/yolo_fish.pt
# 5. Update src/realtime.py default --detector from "yolov8n.pt" to "weights/yolo_fish.pt"
```